In [1]:
# 1. Navigate to the working directory
%cd /kaggle/working/

# 2. Re-clone the LivePortrait repository
!git clone https://github.com/KwaiVGI/LivePortrait
%cd LivePortrait

# 3. Install dependencies and the specific CUDA 12 ONNX runtime
!pip install -r requirements.txt
!pip install insightface 
!pip uninstall -y onnxruntime-gpu
!pip install onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/

# 4. Re-download the AI weights
!pip install -q huggingface_hub
!huggingface-cli download KwaiVGI/LivePortrait --local-dir pretrained_weights --local-dir-use-symlinks False

print("✅ Workspace rebuilt successfully! You are ready to run the randomizer.")

/kaggle/working
Cloning into 'LivePortrait'...
remote: Enumerating objects: 1092, done.
remote: Counting objects: 100% (309/309), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 1092 (delta 270), reused 246 (delta 246), pack-reused 783 (from 3)
Receiving objects: 100% (1092/1092), 38.77 MiB | 29.11 MiB/s, done.
Resolving deltas: 100% (555/555), done.
/kaggle/working/LivePortrait
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.5/881.5 kB 21.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 9.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a wh

In [2]:
import os
import shutil

# 1. The locked folder where your videos currently are
# (Copied exactly from your error message)
READ_ONLY_DRIVING_DIR = '/kaggle/input/datasets/ayushi6/driving-videos/driving vids'

# 2. The new, unlocked folder we are creating
WRITABLE_DRIVING_DIR = '/kaggle/working/driving_videos_writable'

os.makedirs(WRITABLE_DRIVING_DIR, exist_ok=True)

print("Copying driving videos to a writable folder...")

# 3. Copy the files over
copied_count = 0
for f in os.listdir(READ_ONLY_DRIVING_DIR):
    if f.lower().endswith('.mp4'):
        src = os.path.join(READ_ONLY_DRIVING_DIR, f)
        dst = os.path.join(WRITABLE_DRIVING_DIR, f)
        shutil.copy2(src, dst)
        copied_count += 1

print(f"✅ Successfully copied {copied_count} videos!")
print(f"Your new driving directory is: {WRITABLE_DRIVING_DIR}")

Copying driving videos to a writable folder...
✅ Successfully copied 5 videos!
Your new driving directory is: /kaggle/working/driving_videos_writable


In [3]:
import os
import random
import subprocess
from tqdm import tqdm

# --- CONFIGURATION ---
DRIVING_DIR = '/kaggle/working/driving_videos_writable' 
IMAGE_DIR = '/kaggle/input/datasets/ayushi6/downsampled-images/50_downsampled'
OUTPUT_DIR = '/kaggle/working/finalvideos'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Gather all files
valid_img_exts = ('.jpg')
images = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(valid_img_exts)]
videos = [f for f in os.listdir(DRIVING_DIR) if f.lower().endswith('.mp4')]

if not videos:
    print("❌ Error: No .mp4 videos found in the driving directory!")
else:
    # 2. Shuffle the deck of images completely randomly
    random.shuffle(images)
    
    # 3. Calculate how many images each video gets
    num_videos = len(videos)
    images_per_video = len(images) // num_videos
    
    print(f"Found {len(images)} images and {num_videos} driving videos.")
    print(f"Assigning roughly {images_per_video} random images to each video...\n")
    
    # 4. The Distribution & Inference Loop
    for i, video_name in enumerate(videos):
        video_path = os.path.join(DRIVING_DIR, video_name)
        
        # Slice the shuffled list to get this video's specific batch
        start_idx = i * images_per_video
        
        # If it's the very last video, give it all remaining images 
        # (in case the division wasn't perfectly even)
        if i == num_videos - 1:
            end_idx = len(images)
        else:
            end_idx = (i + 1) * images_per_video
            
        assigned_images = images[start_idx:end_idx]
        
        print(f"--- Processing Video {i+1}/{num_videos}: {video_name} ---")
        print(f"Animating {len(assigned_images)} faces with this motion...")
        
        # Run LivePortrait for this specific batch
        for img_name in tqdm(assigned_images):
            img_path = os.path.join(IMAGE_DIR, img_name)
            
            cmd = [
                "python", "inference.py",
                "-s", img_path,
                "-d", video_path,
                "--output_dir", OUTPUT_DIR
            ]
            
            process = subprocess.run(cmd, capture_output=True, text=True, cwd='/kaggle/working/LivePortrait')
            
            if process.returncode != 0:
                print(f"\n❌ Error on {img_name}: {process.stderr.splitlines()[-1]}")

    print("\n✅ All randomized batches are complete!")

Found 49 images and 5 driving videos.
Assigning roughly 9 random images to each video...

--- Processing Video 1/5: speaking.mp4 ---
Animating 9 faces with this motion...


100%|██████████| 9/9 [02:47<00:00, 18.57s/it]


--- Processing Video 2/5: left_tilt.mp4 ---
Animating 9 faces with this motion...


100%|██████████| 9/9 [03:01<00:00, 20.18s/it]


--- Processing Video 3/5: blinking eyes.mp4 ---
Animating 9 faces with this motion...


...

--- Processing Video 4/5: no.mp4 ---
Animating 9 faces with this motion...


100%|██████████| 9/9 [04:30<00:00, 30.08s/it]


--- Processing Video 5/5: tilting.mp4 ---
Animating 13 faces with this motion...


100%|██████████| 13/13 [03:55<00:00, 18.15s/it]


✅ All randomized batches are complete!


In [4]:
import os
import zipfile
from IPython.display import FileLink, display

folder_to_zip = '/kaggle/working/finalvideos'
zip_filename = '/kaggle/working/videos_output.zip' # Note: We include .zip here now

# 1. Create the zip file, filtering out specific files
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_to_zip):
        for file in files:
            # The exact filter: skip if it ends with _contact.mp4
            if file.endswith('_contact.mp4'):
                continue
            
            # Get the full path and the relative path (to keep folder structure clean)
            full_path = os.path.join(root, file)
            relative_path = os.path.relpath(full_path, folder_to_zip)
            
            # Add to zip
            zipf.write(full_path, relative_path)

print(f"Successfully zipped '{folder_to_zip}' into '{zip_filename}'")
print("Excluded all files ending in '_contact.mp4'")

# 2. Create the clickable download link
display(FileLink(r'downloaded_output.zip'))

Successfully zipped '/kaggle/working/finalvideos' into '/kaggle/working/videos_output.zip'
Excluded all files ending in '_contact.mp4'


/kaggle/working/LivePortrait/downloaded_output.zip